<a href="https://colab.research.google.com/github/Apekshagaonkar/ai-systems-notebooks/blob/main/week6_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 — Retrieval-Augmented Generation (RAG)
### Turning a closed-book exam into an open-book exam

Last week we learned an LLM predicts *likely* text, and sometimes makes things up.
This week we fix that for **our own data**: we let the model **look things up** before it answers.

Our story: **SkyCast**, a (made-up) weather-app company. Its staff and customers keep asking a chatbot
questions about SkyCast's own rules: time off, refunds, alerts. The LLM has never seen those documents.


**No API key needed.** A free *pretend* LLM answers, and the search uses a free embedding model that runs inside Colab.

## Setup — run this first
Installs the libraries, writes SkyCast's documents into a `data/` folder, and picks the LLM:
the **real** OpenAI model if you've added a key, otherwise the free **pretend** one.

*Optional:* to use a real key in Colab, click 🔑 (Secrets) on the left, add `OPENAI_API_KEY`, and turn on notebook access.

In [ ]:
!pip -q install sentence-transformers openai

import os, re, json, time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)

MODEL = "gpt-4o"   # change it if your provider has retired this model

# ---- SkyCast's private documents (the model has never seen these) ----
DOCS = {
    "paid_time_off.txt": """Paid Time Off Guidelines

Full-time employees receive 18 days of paid time off each year. Up to 5 unused days roll over into the next year.

Requests must be submitted in the HR portal at least two weeks in advance.""",

    "auto_coverage.txt": """Auto Coverage

Employees who drive their own vehicle for SkyCast business are protected by the company's auto coverage. Accidents during work trips are covered up to $50,000.

Mileage is reimbursed at 70 cents per mile. Submit trip logs to Finance by the 5th of each month.""",

    "remote_work.txt": """Remote Work Guidelines

Employees may work from home up to three days a week. Core collaboration hours are 10am to 3pm Pacific time.

SkyCast provides a $500 home office allowance for a desk, chair or monitor.""",

    "resolving_issues.txt": """Resolving App Issues

When a customer reports a problem in the SkyCast app, engineers first reproduce it on a test device. Next, they write a failing test that captures the problem.

The change is shipped behind a feature flag and rolled out to 5% of users before everyone gets it.""",

    "refunds.txt": """Refunds and Cancellations

SkyCast Premium subscribers can request a full refund within 14 days of purchase. After 14 days you can still cancel to stop the next payment.

Refunds go back to the original payment method within 5 business days.""",

    "privacy.txt": """Location Data and Privacy

SkyCast uses your location only to show local weather. We never sell location data to advertisers.

You can delete your location history at any time in Settings.""",

    "weather_alerts.txt": """Severe Weather Alerts

Premium subscribers receive push alerts for thunderstorms, floods and heat waves in their saved locations.

Alerts are sent as soon as the national weather service issues a warning, usually 15 to 30 minutes before the storm arrives.""",
}
Path("data").mkdir(exist_ok=True)
for name, text in DOCS.items():
    Path("data", name).write_text(text)

# ---- small helpers ----
STOPWORDS = set("""a an the is are was were be been to of and or in on at for with by from as it its this that
these those do does did can could i you we they he she my our your their me us what which who how when where
why if then than so not no up out about into over again more most some any each all other such only same too
very just also will would should may might must have has had am get gets many much""".split())

def words(text):
    """Lowercase words, without filler words like 'the' and 'is'."""
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS and len(w) > 1]

NO_ANSWER = "I don't know based on the SkyCast documents."

# ---- the pretend LLM ----
class MockLLM:
    """A pretend LLM with the same shape as the real OpenAI client.
    With documents in the prompt, it answers from the best-matching sentence.
    Without documents, it imitates what LLMs often do: a confident guess."""
    def __init__(self):
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _answer(self, messages):
        user = messages[-1]["content"]
        if "CONTEXT:" in user and "QUESTION:" in user:
            context = user.split("CONTEXT:")[1].split("QUESTION:")[0]
            question = user.split("QUESTION:")[1]
            sentences = []
            for line in context.splitlines():
                line = re.sub(r"^\[\d+\]\s*", "", line.strip())
                line = line.split(": ", 1)[-1]                    # drop the "Title: " prefix
                sentences += [s for s in re.split(r"(?<=[.!?])\s+", line) if len(s.split()) >= 5]
            if not sentences:
                return NO_ANSWER
            q = set(words(question))
            best = max(sentences, key=lambda s: len(q & set(words(s))))
            return f"According to the SkyCast documents: {best}"
        if re.search(r"time off|vacation|holiday|leave", user, re.I):
            return "SkyCast employees get 15 days of paid time off per year, and unused days expire on December 31."
        return "Good question! Most companies handle that case by case, so it depends on your situation."

    def _create(self, model, messages, temperature=1.0, **kwargs):
        reply = self._answer(messages)
        message = SimpleNamespace(role="assistant", content=reply)
        return SimpleNamespace(model="pretend-" + model,
                               choices=[SimpleNamespace(index=0, message=message, finish_reason="stop")])

key = os.environ.get("OPENAI_API_KEY")
try:
    from google.colab import userdata
    key = key or userdata.get("OPENAI_API_KEY")
except Exception:
    pass

if key:
    from openai import OpenAI
    client = OpenAI(api_key=key)
    os.environ["OPENAI_API_KEY"] = key       # LlamaIndex (Chapter 7) reads it from here
    REAL_API = True
else:
    client = MockLLM()
    REAL_API = False

def ask(messages):
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return r.choices[0].message.content

print("Documents written:", sorted(os.listdir("data")))
print("LLM:", f"REAL {MODEL}" if REAL_API else "pretend model (MockLLM), free, no key needed")

Documents written: ['auto_coverage.txt', 'paid_time_off.txt', 'privacy.txt', 'refunds.txt', 'remote_work.txt', 'resolving_issues.txt', 'weather_alerts.txt']
LLM: pretend model (MockLLM), free, no key needed


---
## Chapter 0 — The closed-book exam

An employee asks the chatbot a simple question about SkyCast. We send it to the LLM **with no documents**.

In [ ]:
question = "How many days of paid time off do SkyCast employees get?"

closed_book = [{"role": "user", "content": question}]
print("Q:", question)
print("A:", ask(closed_book))
print()
print("The real answer is in data/paid_time_off.txt:")
print(" ", DOCS["paid_time_off.txt"].splitlines()[2])

Q: How many days of paid time off do SkyCast employees get?
A: SkyCast employees get 15 days of paid time off per year, and unused days expire on December 31.

The real answer is in data/paid_time_off.txt:
  Full-time employees receive 18 days of paid time off each year. Up to 5 unused days roll over into the next year.


---
## Chapter 1 — Three ways to teach a model new knowledge

| Strategy | How it works | Exam analogy | Best when |
|---|---|---|---|
| **Context injection** | Paste the info straight into the prompt | Bring the *whole library* into the exam room | Small, stable facts |
| **Fine-tuning** | Retrain the model on your data | Memorise the textbook before the exam | Teaching style or tone; expensive |
| **RAG** | Look up the relevant pieces at question time | A librarian brings *just the right pages* | Large or changing knowledge |

Context injection works great when the data is tiny. Let's measure how "tiny" our handbook is, and what happens as it grows.

In [ ]:
def rough_tokens(text):
    return len(text) // 4          # about 4 characters per token (Week 5)

handbook = "\n\n".join(DOCS.values())
per_doc = rough_tokens(handbook) / len(DOCS)

print(f"Our handbook: {len(DOCS)} documents ≈ {rough_tokens(handbook):,} tokens per question if we paste it all\n")
for n_docs in [7, 1_000, 10_000]:
    paste_all = int(per_doc * n_docs)
    fits = "fits" if paste_all <= 128_000 else "DOES NOT FIT in a 128,000-token context"
    print(f"{n_docs:>6,} documents → paste everything: {paste_all:>9,} tokens ({fits})")
print(f"\nRAG sends only the 2 best chunks: ≈ {int(per_doc):,} tokens, no matter how big the library gets.")

Our handbook: 7 documents ≈ 418 tokens per question if we paste it all

     7 documents → paste everything:       418 tokens (fits)
 1,000 documents → paste everything:    59,714 tokens (fits)
10,000 documents → paste everything:   597,142 tokens (DOES NOT FIT in a 128,000-token context)

RAG sends only the 2 best chunks: ≈ 59 tokens, no matter how big the library gets.


---
## Chapter 2 — The RAG pattern  *(12–15 min)*

```
           ①  INDEX (once, ahead of time)
   documents ──► chunks ──► searchable store
                                  │
   user question ──────────► ② RETRIEVE the most relevant chunks
                                  │
                       ③ AUGMENT: put the chunks + question in the prompt
                                  │
                       ④ GENERATE: the LLM answers from those chunks ──► answer
```

| Step | What happens | Exam analogy |
|---|---|---|
| **Index** | Chunk the documents and store them so they're searchable | The library catalogue |
| **Retrieve** | Find the chunks most relevant to the question | The librarian fetches pages |
| **Augment** | Inject those chunks into the prompt next to the question | Pages on the desk |
| **Generate** | The LLM answers using the retrieved context | The student writes the answer |

The answer is **grounded** in real documents, not invented.

---
## Chapter 3 — Keyword RAG from scratch

The "Hello World" of RAG: **plain word matching**, no embeddings, no database. *(Kaggle Lesson 2)*

### Step 1 — Load the documents

In [ ]:
def load_documents(folder="data"):
    return {path.name: path.read_text() for path in sorted(Path(folder).glob("*.txt"))}

documents = load_documents()
print(f"Loaded {len(documents)} documents:", list(documents))

Loaded 7 documents: ['auto_coverage.txt', 'paid_time_off.txt', 'privacy.txt', 'refunds.txt', 'remote_work.txt', 'resolving_issues.txt', 'weather_alerts.txt']


### Step 2 — Chunk them

Why not search whole documents? Real documents can be 50 pages long. We want to hand the LLM **the paragraph that answers the question**, not the whole book.

We split each document into **paragraphs**, and glue the document's **title** onto each chunk, so a chunk still makes sense on its own.

In [ ]:
def chunk_document(name, text):
    title, *paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    return [{"source": name, "text": f"{title}: {p}"} for p in paragraphs]

chunks = [c for name, text in documents.items() for c in chunk_document(name, text)]
print(f"{len(documents)} documents → {len(chunks)} chunks\n")
pd.DataFrame(chunks).head(6)

7 documents → 14 chunks



,source,text
0,auto_coverage.txt,Auto Coverage: Employees who drive their own vehicle for SkyCast business are protecte...
1,auto_coverage.txt,Auto Coverage: Mileage is reimbursed at 70 cents per mile. Submit trip logs to Finance...
2,paid_time_off.txt,Paid Time Off Guidelines: Full-time employees receive 18 days of paid time off each ye...
3,paid_time_off.txt,Paid Time Off Guidelines: Requests must be submitted in the HR portal at least two wee...
4,privacy.txt,Location Data and Privacy: SkyCast uses your location only to show local weather. We n...
5,privacy.txt,Location Data and Privacy: You can delete your location history at any time in Settings.


### Step 3 — Search: count matching words

In [ ]:
def keyword_search(query, k=2):
    query_words = set(words(query))
    scored = []
    for chunk in chunks:
        score = len(query_words & set(words(chunk["text"])))
        if score > 0:
            scored.append({"score": score, **chunk})
    return sorted(scored, key=lambda c: c["score"], reverse=True)[:k]

pd.DataFrame(keyword_search("How many days of paid time off do SkyCast employees get?"))

,score,source,text
0,5,paid_time_off.txt,Paid Time Off Guidelines: Full-time employees receive 18 days of paid time off each ye...
1,3,paid_time_off.txt,Paid Time Off Guidelines: Requests must be submitted in the HR portal at least two wee...


### Step 4 — Augment the prompt, then generate

We put the retrieved chunks into the prompt, and tell the model to answer **only** from them.

In [ ]:
SYSTEM = ("You are SkyCast's helpful assistant. Answer ONLY using the CONTEXT. "
          f"If the answer is not in the CONTEXT, reply exactly: {NO_ANSWER}")

def build_prompt(question, found):
    if found:
        context = "\n".join(f"[{i}] {c['text']}" for i, c in enumerate(found, 1))
    else:
        context = "(no matching documents found)"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}]

def rag(question, search, show_prompt=False):
    found = search(question)                       # ② retrieve
    messages = build_prompt(question, found)       # ③ augment
    if show_prompt:
        print(messages[-1]["content"], "\n")
    answer = ask(messages)                         # ④ generate
    sources = sorted({c["source"] for c in found})
    return f"{answer}\n   sources: {sources or 'none'}"

print("WITH the documents (open book):")
print(rag(question, keyword_search, show_prompt=True))

WITH the documents (open book):
CONTEXT:
[1] Paid Time Off Guidelines: Full-time employees receive 18 days of paid time off each year. Up to 5 unused days roll over into the next year.
[2] Paid Time Off Guidelines: Requests must be submitted in the HR portal at least two weeks in advance.

QUESTION: How many days of paid time off do SkyCast employees get? 

According to the SkyCast documents: Full-time employees receive 18 days of paid time off each year.
   sources: ['paid_time_off.txt']


### Try a question the documents can't answer
*(Kaggle Lesson 2: "try a query where the answer is in the documents, then one where it isn't")*

In [ ]:
for q in ["Can I get my money back for SkyCast Premium?",
          "What is the CEO's favourite food?"]:
    print("Q:", q)
    print("A:", rag(q, keyword_search), "\n")

Q: Can I get my money back for SkyCast Premium?
A: According to the SkyCast documents: SkyCast Premium subscribers can request a full refund within 14 days of purchase.
   sources: ['auto_coverage.txt', 'refunds.txt'] 

Q: What is the CEO's favourite food?
A: I don't know based on the SkyCast documents.
   sources: none 



---
## Chapter 4 — Where keywords fall short

Our librarian only finds chunks that contain **the exact words** of the question. But people don't use the handbook's words.

In [ ]:
tests = {
    "car insurance": "auto_coverage.txt",
    "vacation policy": "paid_time_off.txt",
    "how do we fix the bug": "resolving_issues.txt",
}
for query, should_find in tests.items():
    found = keyword_search(query)
    got = [c["source"] for c in found] or ["nothing"]
    print(f"{query!r:<26} should find {should_find:<22} → keyword search found: {got}")

'car insurance'            should find auto_coverage.txt      → keyword search found: ['nothing']
'vacation policy'          should find paid_time_off.txt      → keyword search found: ['nothing']
'how do we fix the bug'    should find resolving_issues.txt   → keyword search found: ['nothing']


---
## Chapter 5 — Semantic search: searching by meaning  

Last week we hand-made tiny 4-number embeddings. Now we load a **real embedding model**, `all-MiniLM-L6-v2`.
It's free, runs inside Colab (no key), and turns any text into **384 numbers** that capture its meaning.

The plan:
1. Embed every chunk **once** (that's our index)
2. Embed the question
3. Find the chunks whose numbers are **closest** (cosine similarity)

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    def embed(texts):
        return np.asarray(embedder.encode(texts, normalize_embeddings=True))
    EMBEDDER = "all-MiniLM-L6-v2 (real meaning-based embeddings)"
except Exception as e:
    # Fallback so the notebook still runs offline. It only counts words, so it is NOT semantic.
    from sklearn.feature_extraction.text import HashingVectorizer
    _hv = HashingVectorizer(n_features=384, alternate_sign=False, norm="l2")
    def embed(texts):
        return _hv.transform(texts).toarray()
    EMBEDDER = f"word-count fallback, NOT semantic (model failed to load: {type(e).__name__})"

chunk_vectors = embed([c["text"] for c in chunks])     # ① index: embed every chunk once
print("Embedder:", EMBEDDER)
print("Index shape:", chunk_vectors.shape, "→", chunk_vectors.shape[0], "chunks ×", chunk_vectors.shape[1], "numbers each")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder: all-MiniLM-L6-v2 (real meaning-based embeddings)
Index shape: (14, 384) → 14 chunks × 384 numbers each


In [ ]:
def semantic_search(query, k=2, min_score=0.25):
    q = embed([query])[0]
    scores = chunk_vectors @ q                         # cosine similarity (vectors are normalised)
    best = np.argsort(-scores)[:k]
    return [{"score": round(float(scores[i]), 2), **chunks[i]} for i in best if scores[i] >= min_score]

for query, should_find in tests.items():
    top = semantic_search(query, k=1, min_score=-1)[0]
    print(f"{query!r:<26} should find {should_find:<22} → semantic search found: {top['source']}  (score {top['score']})")

'car insurance'            should find auto_coverage.txt      → semantic search found: auto_coverage.txt  (score 0.44)
'vacation policy'          should find paid_time_off.txt      → semantic search found: paid_time_off.txt  (score 0.25)
'how do we fix the bug'    should find resolving_issues.txt   → semantic search found: resolving_issues.txt  (score 0.39)


### Semantic RAG

Same `rag()` function as Chapter 3. We only swap **one argument**: the search.

In [ ]:
for q in ["Am I covered if I crash on a work trip?",
          "How much vacation do I get?",
          "What is the CEO's favourite food?"]:
    print("Q:", q)
    print("A:", rag(q, semantic_search), "\n")

Q: Am I covered if I crash on a work trip?
A: According to the SkyCast documents: Accidents during work trips are covered up to $50,000.
   sources: ['auto_coverage.txt'] 

Q: How much vacation do I get?
A: According to the SkyCast documents: Full-time employees receive 18 days of paid time off each year.
   sources: ['auto_coverage.txt', 'paid_time_off.txt'] 

Q: What is the CEO's favourite food?
A: I don't know based on the SkyCast documents.
   sources: none 



### The policy changes — what now?

HR raises time off from 18 to **20 days**. With fine-tuning, we'd retrain the model. With RAG:

In [ ]:
Path("data/paid_time_off.txt").write_text(DOCS["paid_time_off.txt"].replace("18 days", "20 days"))

documents = load_documents()                                                          # reload
chunks = [c for name, text in documents.items() for c in chunk_document(name, text)]  # re-chunk
chunk_vectors = embed([c["text"] for c in chunks])                                    # re-index

print(rag("How many days of paid time off do SkyCast employees get?", semantic_search))

According to the SkyCast documents: Full-time employees receive 20 days of paid time off each year.
   sources: ['paid_time_off.txt', 'refunds.txt']


---
## Chapter 6 — Storing vectors: persistence, pgvector, and speed

Right now our index lives in a Python variable, `chunk_vectors`. Restart Colab and it's **gone**:
we'd have to re-embed every document. For 10,000 documents, that's slow and costs money every time.

The fix: **save the vectors** somewhere permanent. Here's the idea with plain files:

In [ ]:
store = Path("vector_store"); store.mkdir(exist_ok=True)
np.save(store / "vectors.npy", chunk_vectors)
(store / "chunks.json").write_text(json.dumps(chunks))

del chunk_vectors, chunks                        # pretend Colab restarted…

chunk_vectors = np.load(store / "vectors.npy")   # …and load the index back, no re-embedding
chunks = json.loads((store / "chunks.json").read_text())
print("Reloaded", len(chunks), "chunks and their vectors from disk")
print("Search still works:", semantic_search("car insurance", k=1, min_score=-1)[0]["source"])

Reloaded 14 chunks and their vectors from disk
Search still works: auto_coverage.txt


### The same thing in a real database: pgvector

**pgvector** adds a `vector` column type and similarity search to **Postgres**. *(Kaggle Lesson 3, which runs Postgres in Docker on your own laptop.)*

```sql
CREATE EXTENSION vector;

CREATE TABLE documents (
    id        serial PRIMARY KEY,
    source    text,
    content   text,
    embedding vector(384)          -- 384 numbers, like our model
);

-- the 5 chunks closest to the question
SELECT content FROM documents
ORDER BY embedding <=> %s           -- <=> is cosine DISTANCE
LIMIT 5;
```

| Our notebook | pgvector |
|---|---|
| `chunk_vectors` (NumPy array) | `embedding` column |
| `chunks` (list of dicts) | `content`, `source` columns |
| `np.save` / `np.load` | stored in the database, always there |
| `chunk_vectors @ q`, sort **highest** first | `ORDER BY embedding <=> q`, **lowest** first |

`<=>` returns cosine **distance** = 1 − cosine similarity. So *smaller* distance = *more* similar:

In [ ]:
q = embed(["car insurance"])[0]
similarity = chunk_vectors @ q
rows = [{"source": chunks[i]["source"], "similarity": round(float(similarity[i]), 2),
         "distance (<=>)": round(1 - float(similarity[i]), 2)} for i in np.argsort(-similarity)[:3]]
pd.DataFrame(rows)

,source,similarity,distance (<=>)
0,auto_coverage.txt,0.44,0.56
1,auto_coverage.txt,0.43,0.57
2,weather_alerts.txt,0.15,0.85


### Is this fast enough with 10,000 documents?

Our search compares the question with **every** vector. Let's time it with made-up vectors.

In [ ]:
rng = np.random.default_rng(0)
query = rng.standard_normal(384).astype(np.float32)
for n in [1_000, 10_000, 100_000]:
    fake_index = rng.standard_normal((n, 384)).astype(np.float32)
    start = time.perf_counter()
    for _ in range(10):
        top5 = np.argpartition(-(fake_index @ query), 5)[:5]
    ms = (time.perf_counter() - start) / 10 * 1000
    print(f"{n:>8,} chunks → {ms:6.2f} ms per search")

   1,000 chunks →   0.17 ms per search
  10,000 chunks →   1.30 ms per search
 100,000 chunks →  13.10 ms per search


---
## Chapter 7 — LlamaIndex: production RAG in ~10 lines

Everything we built by hand is a common pattern, so frameworks package it.

```python
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine()
response = query_engine.query("What is the vacation policy?")
```

| What we wrote today | What LlamaIndex calls it |
|---|---|
| `load_documents("data")` | `SimpleDirectoryReader("./data").load_data()` |
| `chunk_document()` + `embed()` + `chunk_vectors` | `VectorStoreIndex.from_documents()` (chunks + embeds + stores) |
| `semantic_search()` + `build_prompt()` + `ask()` | `query_engine.query()` |
| `min_score`, the system message, `k=2` | defaults you can change |

It uses OpenAI for embeddings and answers by default, so the cell below only runs with an API key.

In [ ]:
if REAL_API:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-index"], check=True)
    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

    li_documents = SimpleDirectoryReader("./data").load_data()
    li_index = VectorStoreIndex.from_documents(li_documents)
    query_engine = li_index.as_query_engine()
    print(query_engine.query("What is the vacation policy?"))
else:
    print("Skipped: LlamaIndex needs an OPENAI_API_KEY. Try it in Kaggle Lesson 4.")

Skipped: LlamaIndex needs an OPENAI_API_KEY. Try it in Kaggle Lesson 4.


---
## Wrap-up

1. **LLMs only know their training data.** No private data, nothing after the cutoff, and they may bluff.
2. **Three fixes:** context injection (small facts), fine-tuning (style and tone), **RAG** (large or changing knowledge, the default).
3. **RAG = Index → Retrieve → Augment → Generate.** The open-book exam.
4. **Keyword search** is simple but needs the exact words.
5. **Semantic search** compares embeddings, so it finds meaning: *car insurance* → *auto coverage*.
6. **pgvector** stores vectors in Postgres so the index persists. `<=>` is cosine distance.
7. **LlamaIndex** does it all in ~10 lines, and now you know what's inside.

> *Any time someone asks "can we make this LLM aware of our data?", RAG is almost always the answer.*

### Where this fits in our pipeline
By Weeks 10–11, our weather data will live in a database. The same pattern lets someone ask
*"Was last weekend good for a walk in San Jose?"*: retrieve those days' records, put them in the prompt, and let the LLM answer from real data.

### Think about before next time
- Think of a product you use. Could RAG make it better? What documents would you give the model?
- Our chunks were paragraphs. What could go wrong if chunks were much **bigger**? Much **smaller**?
- A user asks something only half-covered by the documents. What should the chatbot say?